<a href="https://colab.research.google.com/github/mhage82/ARI510/blob/main/lab3_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARI 410/510 Lab 3 Starter Notebook

This notebook gives you the building blocks: downloading the data, viewing a meme, and two pretrained **encoders** (BERT for text, ResNet-18 for images) that turn each meme into vectors. Building, training, and comparing the classifiers is up to you (see the assignment PDF).

**Content note:** this dataset contains hateful and offensive content.

**How to use this:** in Colab, *File → Save a copy in Drive*, and switch to a GPU runtime (*Runtime → Change runtime type*). Run everything on a small subset first (section 5).

> ### ⚠️ Compute your embeddings once, then stop using the images
> The dataset is about 3 GB, and running BERT and ResNet over it is the slowest part of this lab. **You should only do it once.** Section 5 saves your embeddings to your Google Drive. After that, every new Colab session should start at **section 6**, which loads the saved embeddings in a few seconds. No download, no GPU needed.
>
> The only reason to download the images again is to *look at* memes (Part 0, and the 410 "look at the memes" analysis).

## 0. Get the data

The dataset is stored with Git LFS, so this download takes a few minutes. Colab deletes files when your session ends, so this download disappears too.

**If you have already saved your full embeddings (section 5), skip to section 6 instead.** You only need to rerun this cell if you want to look at the images again.

In [ ]:
# OLD VERSION
#!git lfs install
#!git clone https://huggingface.co/datasets/neuralcatcher/hateful_memes

# RECOMMENDED UPDATE
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="cs5242-hateful-memes/hateful-memes-data",
    repo_type="dataset",
    local_dir="hateful_memes",
    ignore_patterns=["data/**"],
)

## 1. Load the splits

In [ ]:
import os
import pandas as pd

DATA = "hateful_memes"   # folder created by the git clone above
print(sorted(f for f in os.listdir(DATA) if f.endswith(".jsonl")))

def load_split(name):
    df = pd.read_json(os.path.join(DATA, name), lines=True)
    df["path"] = df["img"].apply(lambda p: os.path.join(DATA, p))
    return df

train = load_split("train.jsonl")
dev   = load_split("dev_seen.jsonl")
test  = load_split("test_seen.jsonl")

for name, df in [("train", train), ("dev", dev), ("test", test)]:
    print(f"{name}: {len(df)} memes, {df['label'].mean():.1%} hateful")

train.head()

## 2. Look at a meme

`show_meme` displays a meme's image with its text and label. Use it for Part 0 of the assignment, and later to look at examples your models get right and wrong.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_meme(row, note=""):
    """Display one meme. `row` is a row of a split, e.g. dev.iloc[0]."""
    img = Image.open(row["path"]).convert("RGB")
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    label = {0: "not hateful", 1: "hateful"}.get(row.get("label"), "unknown")
    plt.title(f"id {row['id']}  |  label: {label} {note}", fontsize=10)
    plt.show()
    print("Text:", row["text"])

show_meme(train.iloc[0])

## 3. Where embeddings get saved

Computing embeddings is the slow part of this lab. This cell mounts Google Drive so they are saved there and survive the end of your Colab session.

In [ ]:
import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/lab3_embeddings"
except ImportError:   # running locally instead of in Colab
    SAVE_DIR = "lab3_embeddings"
os.makedirs(SAVE_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device, "| saving to", SAVE_DIR)

def get_embeddings(name, compute_fn):
    """Load saved embeddings called `name` if they exist; otherwise compute and save them."""
    path = os.path.join(SAVE_DIR, name + ".pt")
    if os.path.exists(path):
        print("loaded", path)
        return torch.load(path)
    emb = compute_fn()
    torch.save(emb, path)
    print("saved", path, tuple(emb.shape))
    return emb

## 4. The pretrained encoders

Neither model below has ever been trained on memes or on hate speech. BERT was trained to predict missing words in books and Wikipedia; ResNet-18 was trained to sort photos into 1,000 ImageNet categories (dogs, cars, mushrooms, ...). We use them **as-is** (no training, `eval()` mode, no gradients) and keep an internal vector instead of their usual output. Your classifiers then learn from those vectors.

### Text: BERT
We keep BERT's vector for the special `[CLS]` token: **768 numbers per text**.

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()

def embed_texts(texts, batch_size=32):
    """List of strings -> tensor of shape [len(texts), 768]."""
    out = []
    for i in range(0, len(texts), batch_size):
        inputs = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                           max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out.append(bert(**inputs).last_hidden_state[:, 0, :].cpu())
    return torch.cat(out)

print(embed_texts(train["text"].head(4).tolist()).shape)   # expect [4, 768]

### Images: ResNet-18
ResNet-18's last layer turns a 512-number vector into scores for the 1,000 ImageNet categories. We replace that layer with `Identity()` (a layer that does nothing), so the model outputs the **512-number vector** instead. Each set of pretrained weights comes with the image preprocessing it expects (resizing, cropping, normalizing), which we apply to every image.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights)
resnet.fc = torch.nn.Identity()          # remove the final classification layer
resnet = resnet.to(device).eval()
preprocess = weights.transforms()         # the preprocessing these weights expect

def embed_images(paths, batch_size=64):
    """List of image paths -> tensor of shape [len(paths), 512]."""
    out = []
    for i in range(0, len(paths), batch_size):
        # .convert("RGB") handles PNGs with a transparency channel
        batch = torch.stack([preprocess(Image.open(p).convert("RGB"))
                             for p in paths[i:i + batch_size]]).to(device)
        with torch.no_grad():
            out.append(resnet(batch).cpu())
    return torch.cat(out)

print(embed_images(train["path"].head(4).tolist()).shape)   # expect [4, 512]

## 5. Compute and save all the embeddings (once!)

Start with `SUBSET = 200` to make sure everything works end to end. Then set `SUBSET = None` and rerun to embed everything (the image step for the training set takes the longest; a few minutes on a GPU). Because of `get_embeddings`, anything already saved is loaded instead of recomputed.

Note: the file names include the subset size, so subset and full runs don't overwrite each other. This cell also saves each split's ids, texts, and labels as a small CSV, so you can work without the original data files later.

In [ ]:
SUBSET = 200   # set to None for the full dataset

splits = {"train": train, "dev": dev, "test": test}
if SUBSET is not None:
    splits = {name: df.head(SUBSET) for name, df in splits.items()}
tag = "all" if SUBSET is None else str(SUBSET)

X_text, X_image, y = {}, {}, {}
for name, df in splits.items():
    X_text[name]  = get_embeddings(f"{name}_text_{tag}",  lambda: embed_texts(df["text"].tolist()))
    X_image[name] = get_embeddings(f"{name}_image_{tag}", lambda: embed_images(df["path"].tolist()))
    y[name] = torch.tensor(df["label"].values)
    df[["id", "text", "label"]].to_csv(os.path.join(SAVE_DIR, f"{name}_meta_{tag}.csv"), index=False)

for name in splits:
    print(name, tuple(X_text[name].shape), tuple(X_image[name].shape), tuple(y[name].shape))
print("Saved to", SAVE_DIR, ":", sorted(os.listdir(SAVE_DIR)))

Once this has run with `SUBSET = None`, check that the files above are in your Google Drive. **From now on, you never need to recompute them.**

## 6. Starting a new session? Load your saved embeddings here

Once section 5 has run on the full dataset, start every new Colab session **here** instead of at the top. This cell only reads the files saved in your Google Drive: no dataset download, no BERT or ResNet, and no GPU needed.

In [ ]:
import os, torch, pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/lab3_embeddings"
except ImportError:
    SAVE_DIR = "lab3_embeddings"

tag = "all"   # use "200" to load the subset instead
X_text, X_image, y, meta = {}, {}, {}, {}
for name in ["train", "dev", "test"]:
    X_text[name]  = torch.load(os.path.join(SAVE_DIR, f"{name}_text_{tag}.pt"))
    X_image[name] = torch.load(os.path.join(SAVE_DIR, f"{name}_image_{tag}.pt"))
    meta[name]    = pd.read_csv(os.path.join(SAVE_DIR, f"{name}_meta_{tag}.csv"))   # id, text, label
    y[name] = torch.tensor(meta[name]["label"].values)
    print(name, tuple(X_text[name].shape), tuple(X_image[name].shape), tuple(y[name].shape))

## 7. Your turn

You now have, for each split (`"train"`, `"dev"`, `"test"`):
- `X_text[split]`: BERT vectors, shape `[n, 768]`
- `X_image[split]`: ResNet vectors, shape `[n, 512]`
- `y[split]`: labels (1 = hateful, 0 = not hateful)

The rows line up: row *i* of each tensor is the same meme (and, if you loaded from section 6, the same row of `meta[split]`, so you can look up a meme's id and text). Next steps (see the assignment for details): build the classifier, run the sanity checks, train text-only and image-only models, then early and late fusion, and compare.

Keep your work log going as you work (see the Reflection section of the assignment).